# N057 · 二叉堆、建堆与优先队列

**目标：** 实现堆不变量并区分建堆与逐个插入成本。

**先修：** N040, N009, N014。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 完全二叉树；sift_up/down；heapify；最小/最大堆；稳定次序键。

**配套讲解来源：** [同名逐章意见](../../comment/057_heap_fundamentals.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要从零实现一个"优先队列"：一种容器，你不停地往里扔数，它每次都把当前最小的那个数吐给你。Python 标准库里有现成的 `heapq`，但本章偏不用它，而是自己手写一个 `MinHeap`，把堆的底层机制彻底搞明白。

具体要回答三个问题：

1. 堆到底长什么样？——它是一棵用数组存的完全二叉树，而且父亲永远不大于孩子。
2. 插入一个新数、或者拿走堆顶最小的数之后，秩序被破坏了，怎么修回来？
3. 给你一整堆数，是"一个一个插入"建堆快，还是"先随便摆好、再从后往前整体修理"快？答案是后者快一个数量级，这也是本章标题里"区分建堆与逐个插入成本"的意思。

本章还附带一个应用题（就是 LeetCode 1046"最后一块石头的重量"）：地上有一堆石头，每一轮挑最重的两块去相撞——两块一样重就同归于尽，不一样重就只剩"重减轻"那么重的一小块。撞到只剩 0 块或 1 块为止，返回最后剩下的重量（一块不剩返回 0）。

具体到数字的小例子：输入 `[2,7,4,1,8,1]`，输出 `1`。手算过程如下：

- 第 1 轮：最重的两块是 8 和 7，相撞剩下 1，石头变成 {4,2,1,1,1}；
- 第 2 轮：4 和 2 相撞，剩下 2，石头变成 {2,1,1,1}；
- 第 3 轮：2 和 1 相撞，剩下 1，石头变成 {1,1,1}；
- 第 4 轮：1 和 1 相撞，同归于尽，石头变成 {1}；
- 只剩一块重量为 1 的石头，所以答案是 1。

这道题每一轮都要"挑出当前最重的两块"，这正是最大堆最擅长干的事。

## 2. 基础知识：先读懂这些词

- **完全二叉树**：一棵从上到下、从左到右依次填满的二叉树，只有最下面一层可能在右边缺几个节点。它的好处是不需要指针：直接用数组存，第 `i` 个位置的两个孩子一定在 `2i+1` 和 `2i+2`，父亲一定在 `(i-1)//2`。这三个公式本章会反复出现，你可以把它们当成堆的"门牌号规则"。
- **最小堆 / 最大堆**：最小堆（小根堆）要求每个父亲都不大于自己的孩子，所以全堆最小值一定待在堆顶（数组 0 号位）。最大堆（大根堆）反过来，堆顶是最大值。
- **堆不变量**：就是"每个父亲不大于自己的孩子"这条规矩。堆的所有操作，目标都是在改动之后让这条规矩仍然成立。
- **sift_up（上浮）**：插入新元素时的修复动作。把新数放在数组末尾，然后不断和父亲比：它比父亲小就交换，一路"浮"上去，直到不比父亲小、或者浮到了根。
- **sift_down（下沉）**：删除堆顶后的修复动作。把数组最后一个数搬到堆顶，然后不断和两个孩子里较小的那个比：它比那个孩子大就交换，一路"沉"下去，直到比两个孩子都小、或者沉到了叶子。
- **heapify（自底向上建堆）**：把一个乱序数组整体变成合法的堆。做法是从最后一个非叶子节点开始，倒着往前逐个执行下沉。它只需 O(n) 时间，比逐个插入的 O(n log n) 便宜。
- **优先队列**：一个抽象概念，指支持"插入元素"和"取出当前最优先元素"两种操作的队列。堆是实现优先队列的标准手段，所以很多人把两个词混着用。
- **负号封装最大堆**：本章手写的是最小堆，想模拟最大堆，就把每个数取负存进去，取出来时再取负还原。例如 8 存成 -8，-8 在最小堆里最靠顶，弹出再取负就变回 8。

## 3. 思路推导：从小例子开始

Step 1：把"完全二叉树"压平成数组。我们根本不建树对象，只维护一个列表 `data`，位置 `i` 的孩子在 `2i+1`、`2i+2`，父亲在 `(i-1)//2`。这样父子关系靠下标算出来，省内存也省代码。

Step 2：插入（push）就用上浮。新数先追加到列表末尾（树的最下一个空位），因为它只可能比父亲小而违规，而违规只会影响"它到根"这一条祖先链，所以一路向上换即可。

Step 3：删除堆顶（pop）就用下沉。记下堆顶答案，把列表最后一个数挖出来填进 0 号位，再让它和较小的孩子比较、一路向下换。选"较小的孩子"交换是为了不让大数压到小数上面。

Step 4：批量建堆（heapify）用自底向上下沉。从最后一个非叶子下标 `len(a)//2-1` 开始倒着走到 0，对每个位置做一次下沉。为什么快？因为一半的节点是叶子（下沉 0 步），四分之一的节点只可能下沉 1 步……绝大多数节点离地面很近。

Step 5：最大堆 = 最小堆 + 负号。石头题需要"每轮最重的两块"，就把所有重量取负塞进最小堆，堆顶的负数绝对值最大，取负还原就是最重的。

手算演示（正好是 notebook 小实例的数据）：依次 push `[5,2,7,1,3]`，看堆数组怎么变：

| 插入值 | 堆数组（push 后） | 发生了什么 |
| --- | --- | --- |
| 5 | `[5]` | 直接放入根 |
| 2 | `[2,5]` | 2 追加后比父亲 5 小，上浮交换 |
| 7 | `[2,5,7]` | 7 比父亲 2 大，原地不动 |
| 1 | `[1,2,7,5]` | 1 先和父亲 5 换，再和新父亲 2 换，两步浮到根 |
| 3 | `[1,2,7,5,3]` | 3 的父亲是 2，3 不小于 2，不动 |

注意最后一行：数组 `[1,2,7,5,3]` 并不是排好序的，它只保证"每个父亲不大于孩子"——下标 1（值 2）的孩子是下标 3（值 5）和下标 4（值 3）。notebook 的第二张小实例表正是这个数组的"下标—父下标—值"对照：`(0,None,1)、(1,0,2)、(2,0,7)、(3,1,5)、(4,1,3)`，你可以拿它检查父 <= 子这条规矩处处成立。

## 4. 核心代码：MinHeap

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class MinHeap:

    def __init__(self, values=()):
        self.data = heapify_bottom_up(list(values))

    def push(self, x):
        self.data.append(x)
        i = len(self.data) - 1
        while i and self.data[i] < self.data[(i - 1) // 2]:
            p = (i - 1) // 2
            self.data[i], self.data[p] = (self.data[p], self.data[i])
            i = p

    def pop(self):
        if not self.data:
            raise IndexError('empty heap')
        answer = self.data[0]
        last = self.data.pop()
        if self.data:
            self.data[0] = last
            i = 0
            while 2 * i + 1 < len(self.data):
                c = 2 * i + 1
                if c + 1 < len(self.data) and self.data[c + 1] < self.data[c]:
                    c += 1
                if self.data[i] <= self.data[c]:
                    break
                self.data[i], self.data[c] = (self.data[c], self.data[i])
                i = c
        return answer
```

### 逐段读懂代码

### 1. `heapify_bottom_up`：自底向上建堆（内嵌下沉函数）

```python
def heapify_bottom_up(a):

    def sift(i):
        while 2 * i + 1 < len(a):
            child = 2 * i + 1
            if child + 1 < len(a) and a[child + 1] < a[child]:
                child += 1
            if a[i] <= a[child]:
                break
            a[i], a[child] = (a[child], a[i])
            i = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i)
    return a
```

- `sift(i)` 是"让下标 i 的数沉到该在的位置"。循环条件 `2*i+1<len(a)` 表示"i 还有孩子"，没孩子就是叶子，自然结束。
- `child=2*i+1` 先默认选左孩子；`if child+1<len(a) and a[child+1]<a[child]: child+=1` 是说：如果右孩子存在且更小，就改选右孩子。我们要和孩子中"较小的那个"交换。
- `if a[i]<=a[child]: break` 表示父亲已经不大于较小的孩子了，规矩满足，收工。
- 交换那行 `a[i],a[child]=a[child],a[i]; i=child` 是 Python 的双赋值交换，交换后让 `i` 跟着降到孩子位置，继续下一轮比较。
- 最后一行 for 循环是关键：`range(len(a)//2-1,-1,-1)` 从"最后一个非叶子节点"倒着走到 0。`len(a)//2-1` 恰好是最后一个有孩子的下标（例如 n=5 时是 2，即值 7 那个位置，它的孩子在 5 和 6——越界说明 5、6 不存在时它就是叶子）。倒着走保证了"先修好子堆，再修父节点"。

### 2. `MinHeap` 类：优先队列本体

```python
class MinHeap:

    def __init__(self, values=()):
        self.data = heapify_bottom_up(list(values))
```

构造函数接受任意可迭代对象（不传就是空堆），先 `list()` 拷贝一份（不污染调用方的列表），再用 O(n) 建堆一次性整理好。

```python
def push(self,x):
        self.data.append(x); i=len(self.data)-1
        while i and self.data[i]<self.data[(i-1)//2]:
            p=(i-1)//2; self.data[i],self.data[p]=self.data[p],self.data[i]; i=p
```

- `append(x)` 把新数放到树的最下最右空位；`i=len(self.data)-1` 是它的下标。
- `while i and ...` 里的 `i` 兼职当"还没到根"的判断（`i=0` 时是根，没有父亲，循环必须停）；条件后半句"我比父亲小"才继续换。
- 交换后 `i=p`，跳到父亲位置继续比，最多一路换到根。

```python
def pop(self):
        if not self.data: raise IndexError('empty heap')
        answer=self.data[0]; last=self.data.pop()
        if self.data:
            self.data[0]=last; i=0
            while 2*i+1<len(self.data):
                c=2*i+1
                if c+1<len(self.data) and self.data[c+1]<self.data[c]: c+=1
                if self.data[i]<=self.data[c]: break
                self.data[i],self.data[c]=self.data[c],self.data[i]; i=c
        return answer
```

- `if not self.data: raise IndexError` 是在挡"空堆还 pop"这种非法操作，与其悄悄给错值，不如大声报错。
- `answer=self.data[0]` 先把堆顶最小值存好；`last=self.data.pop()` 把数组最后一个数摘下来。
- `if self.data:` 这个判断很细：如果摘掉之后堆空了（原来只有 1 个元素），就不能再把 `last` 塞回去，直接返回答案即可；否则把 `last` 放到堆顶，开始下沉。
- 下沉循环和 `sift` 的逻辑一模一样：选较小孩子、能停就停、不能停就交换并继续。
- 为什么要"摘尾补顶"而不是"整体前移一位"？因为整体前移是 O(n)，而摘尾补顶加下沉只有 O(log n)。

### 3. `last_stone_weight`：负号最大堆打石头

```python
def last_stone_weight(stones):
    heap = MinHeap((-x for x in stones))
    while len(heap.data) > 1:
        a = -heap.pop()
        b = -heap.pop()
        if a != b:
            heap.push(-(a - b))
    return -heap.pop() if heap.data else 0
```

- 第一行用生成器把每块石头重量取负塞进最小堆，等效做出了最大堆。
- `while len(heap.data)>1`：还剩至少两块就继续撞。`a` 是弹出的最重块（取负还原），`b` 是次重块。
- `if a!=b: heap.push(-(a-b))`：两块不等才剩差值块（注意先把差值取负再入堆）；相等就什么都不放回去，等于两块同归于尽。
- 最后一行是 Python 三元表达式：堆里还有一块就返回它的重量，一块不剩（比如两块等重撞光了）就返回 0。

## 5. 分段实现：按顺序运行

**本章接口：** `MinHeap`、`heapify_bottom_up(a)`、`last_stone_weight(stones)`

### heapify_bottom_up

In [1]:
def heapify_bottom_up(a):

    def sift(i):
        while 2 * i + 1 < len(a):
            child = 2 * i + 1
            if child + 1 < len(a) and a[child + 1] < a[child]:
                child += 1
            if a[i] <= a[child]:
                break
            a[i], a[child] = (a[child], a[i])
            i = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i)
    return a

### MinHeap

In [2]:
class MinHeap:

    def __init__(self, values=()):
        self.data = heapify_bottom_up(list(values))

    def push(self, x):
        self.data.append(x)
        i = len(self.data) - 1
        while i and self.data[i] < self.data[(i - 1) // 2]:
            p = (i - 1) // 2
            self.data[i], self.data[p] = (self.data[p], self.data[i])
            i = p

    def pop(self):
        if not self.data:
            raise IndexError('empty heap')
        answer = self.data[0]
        last = self.data.pop()
        if self.data:
            self.data[0] = last
            i = 0
            while 2 * i + 1 < len(self.data):
                c = 2 * i + 1
                if c + 1 < len(self.data) and self.data[c + 1] < self.data[c]:
                    c += 1
                if self.data[i] <= self.data[c]:
                    break
                self.data[i], self.data[c] = (self.data[c], self.data[i])
                i = c
        return answer

### last_stone_weight

In [3]:
def last_stone_weight(stones):
    heap = MinHeap((-x for x in stones))
    while len(heap.data) > 1:
        a = -heap.pop()
        b = -heap.pop()
        if a != b:
            heap.push(-(a - b))
    return -heap.pop() if heap.data else 0

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

h=MinHeap(); rows=[]
for x in [5,2,7,1,3]:
    h.push(x); rows.append((x,h.data[:]))
show_table(['插入值','堆数组'],rows)
show_table(['数组索引','父索引','值'],[(i,(i-1)//2 if i else None,v) for i,v in enumerate(h.data)])

插入值,堆数组
5,[5]
2,"[2, 5]"
7,"[2, 5, 7]"
1,"[1, 2, 7, 5]"
3,"[1, 2, 7, 5, 3]"


数组索引,父索引,值
0,None,1
1,0,2
2,0,7
3,1,5
4,1,3


## 7. 正确性、适用条件与复杂度

子堆先合法，sift选择较小孩子交换，只需继续修复被替换的子树。高度h的节点数量约n/2^(h+1)，总下沉工作Σh·n/2^(h+1)=O(n)。

**代价：** 建堆O(n)，push/pop最坏O(log n)，空间O(n)；石头模拟O(n log n)。

### 展开理解

先说建堆为什么是 O(n)。直觉是：越靠近树底的节点越多，而它们离"地面"（叶子层）越近，下沉距离越短。拿 n=1000 举例：大约 500 个是叶子，压根不用下沉；250 个最多沉 1 层；125 个最多沉 2 层……总工作量约等于 500×0 + 250×1 + 125×2 + …… 这个级数加起来大约就是 1000 的量级，所以是 O(n)，而不是直觉上的 O(n log n)。反过来，"逐个 push 建堆"每个插入都可能从底浮到顶（log n 层），n 个元素就是 O(n log n)——这就是本章开头那句"自底向上建堆比逐个插入更便宜"的完整含义。

再说 push 和 pop 为什么是 O(log n)。上浮和下沉每比较一次，位置就升/降一层，而完全二叉树一共只有 log₂n 层。n=10⁶ 时 log₂n 约 20，也就是说一百万个数的堆，插一个、删一个都只要大约 20 次比较，一瞬间就算完。石头题每轮撞两次 pop、至多一次 push，共 n-1 轮，所以整体 O(n log n)；n=10⁵ 时大约就是百万级的基本操作，毫无压力。

下沉为什么选"较小的孩子"交换？因为交换后新的父节点必须同时压得住两个孩子；如果跟较大的孩子换，较小的孩子升上来仍然比它小（不对，是被换下去的那个现在成了较大孩子的父亲……准确说：跟较大孩子交换后，原来较小的孩子会被压在新父亲下面而违规）。一句话：跟小的换，两个都服；跟大的换，小的不服。

## 8. 单元测试：每个用例在检查什么

notebook 的测试 cell 先做了一场 1000 回合的"影子对决"：每回合随机决定是 push 一个 -50 到 50 的随机数，还是 pop 一次；`MinHeap` 的每一步输出都和标准库 `heapq` 维护的参考堆逐个对比，保证行为完全一致。而且每回合结束后都有一条 `assert all(h.data[(i-1)//2]<=h.data[i] ...)`，直接把"父 <= 子"这条堆不变量在每个下标上检查一遍——这比只对答案更严格，等于在测"内部状态始终合法"。

接着 `assert [h.pop() for _ in ...]==sorted(ref)`：把堆整个倒空，验证弹出的顺序正好是升序排序，也就是"优先队列确实每次给最小值"。

`assert last_stone_weight([2,7,4,1,8,1])==1` 是第一节手算过的正常用例；`assert last_stone_weight([3])==3` 是边界用例——只有一块石头，没人可撞，原样返回。

In [5]:
import heapq

from random import Random

rng = Random(57)

h = MinHeap()

ref = []

for _ in range(1000):
    if not ref or rng.randrange(3):
        x = rng.randrange(-50, 51)
        h.push(x)
        heapq.heappush(ref, x)
    else:
        assert h.pop() == heapq.heappop(ref)
    assert all((h.data[(i - 1) // 2] <= h.data[i] for i in range(1, len(h.data))))

assert [h.pop() for _ in range(len(h.data))] == sorted(ref)

assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1

assert last_stone_weight([3]) == 3

print('N057: 所有本章断言通过')

N057: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 证明自底向上建堆的线性成本。

**练习 2：** 用负号封装最大堆并说明数据类型限制。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- 练习 1（证明自底向上建堆的线性成本）：把第三节"500 个叶子、250 个沉一层"的账正式写成求和：高度为 h 的节点约 n/2^(h+1) 个，每个最多沉 h 层，总代价是 Σ h·n/2^(h+1)。提示：利用"每项乘 h 但每层人数减半"的收敛性，说明和被 cn 常数倍封顶。写清楚输入（任意数组）和输出（合法堆）是什么，并拿 `[5,2,7,1,3]` 手算一遍每个节点的下沉步数来对照。
- 练习 2（负号封装最大堆及其限制）：`last_stone_weight` 已经示范了取负。要说明的"数据类型限制"是：这套技巧只对"可取负的数值"有效；如果元素是字符串、自定义对象，取负没有定义，必须换"传入 key/比较函数"的方案。提示：你可以试着写一个 `MaxHeap(key=...)`，把"取负"推广成"用 key 把任意比较需求映射成小根堆的比较"。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def heapify_bottom_up(a):

    def sift(i):
        while 2 * i + 1 < len(a):
            child = 2 * i + 1
            if child + 1 < len(a) and a[child + 1] < a[child]:
                child += 1
            if a[i] <= a[child]:
                break
            a[i], a[child] = (a[child], a[i])
            i = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i)
    return a

class MinHeap:

    def __init__(self, values=()):
        self.data = heapify_bottom_up(list(values))

    def push(self, x):
        self.data.append(x)
        i = len(self.data) - 1
        while i and self.data[i] < self.data[(i - 1) // 2]:
            p = (i - 1) // 2
            self.data[i], self.data[p] = (self.data[p], self.data[i])
            i = p

    def pop(self):
        if not self.data:
            raise IndexError('empty heap')
        answer = self.data[0]
        last = self.data.pop()
        if self.data:
            self.data[0] = last
            i = 0
            while 2 * i + 1 < len(self.data):
                c = 2 * i + 1
                if c + 1 < len(self.data) and self.data[c + 1] < self.data[c]:
                    c += 1
                if self.data[i] <= self.data[c]:
                    break
                self.data[i], self.data[c] = (self.data[c], self.data[i])
                i = c
        return answer

def last_stone_weight(stones):
    heap = MinHeap((-x for x in stones))
    while len(heap.data) > 1:
        a = -heap.pop()
        b = -heap.pop()
        if a != b:
            heap.push(-(a - b))
    return -heap.pop() if heap.data else 0

# 示例与回归测试
import heapq

from random import Random

rng = Random(57)

h = MinHeap()

ref = []

for _ in range(1000):
    if not ref or rng.randrange(3):
        x = rng.randrange(-50, 51)
        h.push(x)
        heapq.heappush(ref, x)
    else:
        assert h.pop() == heapq.heappop(ref)
    assert all((h.data[(i - 1) // 2] <= h.data[i] for i in range(1, len(h.data))))

assert [h.pop() for _ in range(len(h.data))] == sorted(ref)

assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1

assert last_stone_weight([3]) == 3

print('N057: 所有本章断言通过')

N057: 所有本章断言通过


## 11. 代表题与迁移

- **1046. Last Stone Weight（最后一块石头的重量）**：本章的 `last_stone_weight` 就是它的直接实现，练的是"负号最大堆 + 每轮取两个堆顶"这个组合拳。
- **703. Kth Largest Element in a Stream（数据流中的第 K 大元素）**：下一章 N058 的主角，这里先预告——它练的是"用一个小根堆只保留前 K 大"，是本章优先队列的直接升级。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。